In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.memory import ConversationBufferWindowMemory
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(model = "gemini-2.0-flash")

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Act as helpful AI Assistant"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

memory = ConversationBufferWindowMemory(k = 3, return_messages=True)

C:\Users\Mohd Faizan Umar\AppData\Local\Temp\ipykernel_9704\2065408908.py:7: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k = 3, return_messages=True)


In [7]:
memory.load_memory_variables({})

{'history': []}

In [8]:
from operator import itemgetter

chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(memory.load_memory_variables)
        |
        itemgetter("history")
    )
    | prompt | model
)

In [10]:
user_input = {"input": "What are the first 4 colours of Rainbow"}

response = chain.invoke(user_input)
response.content

'The first four colors of the rainbow are:\n\n1.  **Red**\n2.  **Orange**\n3.  **Yellow**\n4.  **Green**'

In [12]:
memory.load_memory_variables({})

{'history': []}

In [13]:
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({})

{'history': [HumanMessage(content='What are the first 4 colours of Rainbow', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of the rainbow are:\n\n1.  **Red**\n2.  **Orange**\n3.  **Yellow**\n4.  **Green**', additional_kwargs={}, response_metadata={})]}

In [14]:
user_input = {"input": "And the last 3?"}
response = chain.invoke(user_input)
response.content

'The last three colors of the rainbow are:\n\n1.  **Blue**\n2.  **Indigo**\n3.  **Violet**'

In [15]:
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({})

{'history': [HumanMessage(content='What are the first 4 colours of Rainbow', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of the rainbow are:\n\n1.  **Red**\n2.  **Orange**\n3.  **Yellow**\n4.  **Green**', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='And the last 3?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The last three colors of the rainbow are:\n\n1.  **Blue**\n2.  **Indigo**\n3.  **Violet**', additional_kwargs={}, response_metadata={})]}